# Run scripts as a pipeline job

A pipeline allows you to group multiple steps into one workflow. You can build a pipeline with components. Each component reflects a Python script to run. A component is defined in a YAML file which specifies the script and how to run it. 

## Before you start

You'll need the latest version of the  **azure-ai-ml** package to run the code in this notebook. Run the cell below to verify that it is installed.

> **Note**:
> If the **azure-ai-ml** package is not installed, run `pip install azure-ai-ml` to install it.

## Connect to your workspace

With the required SDK packages installed, now you're ready to connect to your workspace.

To connect to a workspace, we need identifier parameters - a subscription ID, resource group name, and workspace name. Since you're working with a compute instance, managed by Azure Machine Learning, you can use the default values to connect to the workspace.

In [11]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential not work
    credential = InteractiveBrowserCredential()

In [12]:
# Get a handle to workspace
ml_client = MLClient.from_config(credential=credential)

Found the config file in: C:\Users\paymo\Downloads\mslearn-mlops\config.json


## Create the scripts

You'll build a pipeline with two steps:

1. **Prepare the data**: Fix missing data and normalize the data.
1. **Train the model**: Train a logistic regression model.

Run the following cells to create the **src** folder and the two scripts.

In [2]:
import os

# create a folder for the script files
script_folder = 'src'
os.makedirs(script_folder, exist_ok=True)
print(script_folder, 'folder created')

src folder created


In [3]:
%%writefile $script_folder/prep-data.py
# import libraries
import argparse
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

def main(args):
    # read data
    df = get_data(args.input_data)

    cleaned_data = clean_data(df)

    normalized_data = normalize_data(cleaned_data)

    output_df = normalized_data.to_csv((Path(args.output_data) / "diabetes.csv"), index = False)

# function that reads the data
def get_data(path):
    df = pd.read_csv(path)

    # Count the rows and print the result
    row_count = (len(df))
    print('Preparing {} rows of data'.format(row_count))
    
    return df

# function that removes missing values
def clean_data(df):
    df = df.dropna()
    
    return df

# function that normalizes the data
def normalize_data(df):
    scaler = MinMaxScaler()
    num_cols = ['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness','SerumInsulin','BMI','DiabetesPedigree']
    df[num_cols] = scaler.fit_transform(df[num_cols])

    return df

def parse_args():
    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument("--input_data", dest='input_data',
                        type=str)
    parser.add_argument("--output_data", dest='output_data',
                        type=str)

    # parse args
    args = parser.parse_args()

    # return args
    return args

# run script
if __name__ == "__main__":
    # add space in logs
    print("\n\n")
    print("*" * 60)

    # parse args
    args = parse_args()

    # run main function
    main(args)

    # add space in logs
    print("*" * 60)
    print("\n\n")

Writing src/prep-data.py


In [4]:
%%writefile $script_folder/train-model-mlflow.py
# import libraries
import mlflow
import glob
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

def main(args):
    # enable autologging
    mlflow.autolog()

    # read data
    df = get_data(args.training_data)

    # split data
    X_train, X_test, y_train, y_test = split_data(df)

    # train model
    model = train_model(args.reg_rate, X_train, X_test, y_train, y_test)

    eval_model(model, X_test, y_test)

# function that reads the data
def get_data(data_path):

    all_files = glob.glob(data_path + "/*.csv")
    df = pd.concat((pd.read_csv(f) for f in all_files), sort=False)
    
    return df

# function that splits the data
def split_data(df):
    print("Splitting data...")
    X, y = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness',
    'SerumInsulin','BMI','DiabetesPedigree','Age']].values, df['Diabetic'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    return X_train, X_test, y_train, y_test

# function that trains the model
def train_model(reg_rate, X_train, X_test, y_train, y_test):
    mlflow.log_param("Regularization rate", reg_rate)
    print("Training model...")
    model = LogisticRegression(C=1/reg_rate, solver="liblinear").fit(X_train, y_train)

    mlflow.sklearn.save_model(model, args.model_output)

    return model

# function that evaluates the model
def eval_model(model, X_test, y_test):
    # calculate accuracy
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    print('Accuracy:', acc)

    # calculate AUC
    y_scores = model.predict_proba(X_test)
    auc = roc_auc_score(y_test,y_scores[:,1])
    print('AUC: ' + str(auc))

    # plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:,1])
    fig = plt.figure(figsize=(6, 4))
    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')
    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.savefig("ROC-Curve.png") 
    mlflow.log_figure(fig, "ROC-Curve.png")
def parse_args():
    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument("--training_data", dest='training_data',
                        type=str)
    parser.add_argument("--reg_rate", dest='reg_rate',
                        type=float, default=0.01)
    parser.add_argument("--model_output", dest='model_output',
                        type=str)

    # parse args
    args = parser.parse_args()

    # return args
    return args

# run script
if __name__ == "__main__":
    # add space in logs
    print("\n\n")
    print("*" * 60)

    # parse args
    args = parse_args()

    # run main function
    main(args)

    # add space in logs
    print("*" * 60)
    print("\n\n")


Writing src/train-model-mlflow.py


## Define the components

To define the component you need to specify:

- **Metadata**: *name*, *display name*, *version*, *description*, *type* etc. The metadata helps to describe and manage the component.
- **Interface**: *inputs* and *outputs*. For example, a model training component will take training data and the regularization rate as input, and generate a trained model file as output. 
- **Command, code & environment**: the *command*, *code* and *environment* to run the component. Command is the shell command to execute the component. Code usually refers to a source code directory. Environment could be an AzureML environment (curated or custom created), docker image or conda environment.

Run the following cells to create a YAML for each component you want to run as a pipeline step.

In [5]:
%%writefile prep-data.yml
$schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
name: prep_data
display_name: Prepare training data
version: 1
type: command
inputs:
  input_data: 
    type: uri_file
outputs:
  output_data:
    type: uri_folder
code: ./src
environment: azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest
command: >-
  python prep-data.py 
  --input_data ${{inputs.input_data}}
  --output_data ${{outputs.output_data}}

Writing prep-data.yml


In [17]:
%%writefile train-model.yml
$schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
name: train_model
display_name: Train a logistic regression model
version: 1
type: command
inputs:
  training_data: 
    type: uri_folder
  reg_rate:
    type: number
    default: 0.01
outputs:
  model_output:
    type: mlflow_model
code: ./src
environment: azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest
command: >-
  python train-model-mlflow.py 
  --training_data ${{inputs.training_data}} 
  --reg_rate ${{inputs.reg_rate}} 
  --model_output ${{outputs.model_output}} 

Overwriting train-model.yml


## Load the components

Now that you have defined each component, you can load the components by referring to the YAML files. 

In [18]:
from azure.ai.ml import load_component
parent_dir = ""

prep_data = load_component(source=parent_dir + "./prep-data.yml")
train_logistic_regression = load_component(source=parent_dir + "./train-model.yml")

## Build the pipeline

After creating and loading the components, you can build the pipeline. You'll compose the two components into a pipeline. First, you'll want the `prep_data` component to run. The output of the first component should be the input of the second component `train_logistic_regression`, which will train the model.

The `diabetes_classification` function represents the complete pipeline. The function expects one input variable: `pipeline_job_input`. A data asset was created during setup. You'll use the registered data asset as the pipeline input. 

In [20]:
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.dsl import pipeline

#import pipeline decorator from azure.ai.ml.dsl
@pipeline()
def diabetes_classification(pipeline_job_input):
    clean_data = prep_data(input_data=pipeline_job_input)
    train_model = train_logistic_regression(training_data=clean_data.outputs.output_data)

    #These lines basically providing the path URI to the outputs of the components. 
    #The outputs of the pipeline job will be the outputs of the components.
    return {
        "pipeline_job_transformed_data": clean_data.outputs.output_data,
        "pipeline_job_trained_model": train_model.outputs.model_output,
    }

pipeline_job = diabetes_classification(Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-data:1"))

You can retrieve the configuration of the pipeline job by printing the `pipeline_job` object:

In [21]:
print(pipeline_job)

display_name: diabetes_classification
type: pipeline
inputs:
  pipeline_job_input:
    type: uri_file
    path: azureml:diabetes-data:1
outputs:
  pipeline_job_transformed_data:
    type: uri_folder
  pipeline_job_trained_model:
    type: mlflow_model
jobs:
  clean_data:
    type: command
    inputs:
      input_data:
        path: ${{parent.inputs.pipeline_job_input}}
    outputs:
      output_data: ${{parent.outputs.pipeline_job_transformed_data}}
    component:
      $schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
      name: prep_data
      version: '1'
      display_name: Prepare training data
      type: command
      inputs:
        input_data:
          type: uri_file
      outputs:
        output_data:
          type: uri_folder
      command: python prep-data.py  --input_data ${{inputs.input_data}} --output_data
        ${{outputs.output_data}}
      environment: azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest
      code: C:/Users/pay

You can change any parameter of the pipeline job configuration by referring to the parameter and specifying the new value:

In [22]:
# change the output mode
#Here we define the output path for the pipeline job outputs. which are originally the path for output of the components. 
# We are changing the output mode to upload so that the outputs will be uploaded to the default datastore of the workspace.
pipeline_job.outputs.pipeline_job_transformed_data.mode = "upload"
pipeline_job.outputs.pipeline_job_trained_model.mode = "upload"
# set pipeline level compute
pipeline_job.settings.default_compute = "cluster1"
# set pipeline level datastore
pipeline_job.settings.default_datastore = "workspaceblobstore"

# print the pipeline job again to review the changes
print(pipeline_job)

display_name: diabetes_classification
type: pipeline
inputs:
  pipeline_job_input:
    type: uri_file
    path: azureml:diabetes-data:1
outputs:
  pipeline_job_transformed_data:
    mode: upload
    type: uri_folder
  pipeline_job_trained_model:
    mode: upload
    type: mlflow_model
jobs:
  clean_data:
    type: command
    inputs:
      input_data:
        path: ${{parent.inputs.pipeline_job_input}}
    outputs:
      output_data: ${{parent.outputs.pipeline_job_transformed_data}}
    component:
      $schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
      name: prep_data
      version: '1'
      display_name: Prepare training data
      type: command
      inputs:
        input_data:
          type: uri_file
      outputs:
        output_data:
          type: uri_folder
      command: python prep-data.py  --input_data ${{inputs.input_data}} --output_data
        ${{outputs.output_data}}
      environment: azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-c

## Submit the pipeline job

Finally, when you've built the pipeline and configured the pipeline job to run as required, you can submit the pipeline job:

In [23]:
# submit job to workspace
pipeline_job = ml_client.jobs.create_or_update(
    pipeline_job, experiment_name="pipeline_diabetes"
)
pipeline_job

Experiment,Name,Type,Status,Details Page
pipeline_diabetes,jolly_berry_82mh1p29w8,pipeline,JobStatus.NOT_STARTED,Link to Azure Machine Learning studio


In [ ]:
tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri
tracking_uri

'azureml://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demoGroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht'

In [31]:
from mlflow.entities import ViewType
import mlflow
mlflow.set_tracking_uri(tracking_uri)
experiments = mlflow.search_experiments(view_type=ViewType.ALL)
for exp in experiments:
    print(exp.name)

prepare_image
azureml-examples
model
sdk_job_data_from_registry
heart-condition-classifier
heart-disease-classifier
Default
heart-classifier-sample
model-management-examples
mlflow-experiment
quickstart
heart-classifier-kr8n0-4595h
heart-classifier-sdloy
hello-batch-pzhot
Diabetes_Decision_Example_Model_Training_1790803444
bluff_experiment_studio_pipeline
hello-world-parallel-job
pipeline_samples
diabetes-training
sweep-diabetes
pipeline_diabetes


In [44]:
#Getting the last run using order by start time of the run and getting the last run from the list of runs.
experiment = mlflow.get_experiment_by_name("pipeline_diabetes")
last_run = mlflow.search_runs(experiment.experiment_id, order_by=["start_time DESC"], max_results=2, output_format="list")[0]

In [45]:
print(last_run.info.run_id)
print(last_run.data.metrics)
print(last_run.data.params)
print(last_run.data.tags)

664c23a9-10fb-46ea-915c-65b449c474b4
{'training_accuracy_score': 0.7911428571428571, 'training_f1_score': 0.7861995251617786, 'training_log_loss': 0.4343412739951813, 'training_precision_score': 0.7860806711610154, 'training_recall_score': 0.7911428571428571, 'training_roc_auc': 0.8619080177080017, 'training_score': 0.7911428571428571}
{'Regularization rate': '0.01', 'C': '100.0', 'class_weight': 'None', 'dual': 'False', 'fit_intercept': 'True', 'intercept_scaling': '1', 'l1_ratio': 'None', 'max_iter': '100', 'multi_class': 'auto', 'n_jobs': 'None', 'penalty': 'l2', 'random_state': 'None', 'solver': 'liblinear', 'tol': '0.0001', 'verbose': '0', 'warm_start': 'False'}
{'azureml.nodeid': '53043363', 'azureml.pipeline': 'jolly_berry_82mh1p29w8', 'estimator_name': 'LogisticRegression', 'estimator_class': 'sklearn.linear_model._logistic.LogisticRegression', 'mlflow.parentRunId': 'jolly_berry_82mh1p29w8', 'mlflow.rootRunId': 'jolly_berry_82mh1p29w8', 'mlflow.runName': 'train_model', 'mlflow.

In [46]:
#Let;s get all the list of artifacts using MLflow client API. We can use the run_id of the last run to get the list of artifacts.
from mlflow.tracking import MlflowClient
client = MlflowClient(tracking_uri=tracking_uri)

artifacts = client.list_artifacts(last_run.info.run_id)
artifacts

[<FileInfo: file_size=-1, is_dir=False, path='ROC-Curve.png'>,
 <FileInfo: file_size=-1, is_dir=False, path='estimator.html'>,
 <FileInfo: file_size=-1, is_dir=True, path='logs'>,
 <FileInfo: file_size=-1, is_dir=True, path='model'>,
 <FileInfo: file_size=-1, is_dir=True, path='system_logs'>,
 <FileInfo: file_size=-1, is_dir=False, path='training_confusion_matrix.png'>,
 <FileInfo: file_size=-1, is_dir=False, path='training_precision_recall_curve.png'>,
 <FileInfo: file_size=-1, is_dir=False, path='training_roc_curve.png'>,
 <FileInfo: file_size=-1, is_dir=True, path='user_logs'>]

In [ ]:
#Download the model artifact from the last run using mlflow
mlflow.artifacts.download_artifacts(run_id = last_run.info.run_id, artifact_path = "model",dst_path = "./model")

'c:\\Users\\paymo\\Downloads\\mslearn-mlops\\experimentation\\model\\model'

In [40]:
experiment = mlflow.get_experiment_by_name("pipeline_diabetes")
mlflow.search_runs(experiment.experiment_id, filter_string="attributes.status = 'Finished'")

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.training_accuracy_score,metrics.training_log_loss,metrics.training_score,metrics.training_roc_auc,...,params.intercept_scaling,params.dual,tags.azureml.nodeid,tags.mlflow.parentRunId,tags.mlflow.rootRunId,tags.mlflow.user,tags.mlflow.runName,tags.azureml.pipeline,tags.estimator_class,tags.estimator_name
0,f454392e-6866-405c-849f-564c72029ab3,2b2438c7-2d79-4ab3-b977-7a6c7c081224,FINISHED,,2026-10-02 13:07:27.452000+00:00,2026-10-02 13:10:15.537000+00:00,NaN,NaN,NaN,NaN,...,None,None,6da265f9,nice_ear_v2rskym10f,nice_ear_v2rskym10f,lisha dalve,clean_data,nice_ear_v2rskym10f,None,None
1,jolly_berry_82mh1p29w8,2b2438c7-2d79-4ab3-b977-7a6c7c081224,FINISHED,,2026-10-02 13:21:33.876000+00:00,2026-10-02 13:26:00.002000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,None,jolly_berry_82mh1p29w8,lisha dalve,diabetes_classification,None,None,None
2,1cb1dfbe-65ba-41ea-8e60-6fc9ab420310,2b2438c7-2d79-4ab3-b977-7a6c7c081224,FINISHED,,2026-10-02 13:21:34.960000+00:00,2026-10-02 13:21:34.960000+00:00,NaN,NaN,NaN,NaN,...,None,None,fbedb2ba,jolly_berry_82mh1p29w8,jolly_berry_82mh1p29w8,lisha dalve,clean_data,jolly_berry_82mh1p29w8,None,None
3,664c23a9-10fb-46ea-915c-65b449c474b4,2b2438c7-2d79-4ab3-b977-7a6c7c081224,FINISHED,,2026-10-02 13:23:36.841000+00:00,2026-10-02 13:25:57.231000+00:00,0.791143,0.434341,0.791143,0.861908,...,1,False,53043363,jolly_berry_82mh1p29w8,jolly_berry_82mh1p29w8,lisha dalve,train_model,jolly_berry_82mh1p29w8,sklearn.linear_model._logistic.LogisticRegression,LogisticRegression


In [50]:
import pandas as pd

df = pd.read_csv("./data/diabetes-dev.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   PatientID               10000 non-null  int64  
 1   Pregnancies             10000 non-null  int64  
 2   PlasmaGlucose           10000 non-null  int64  
 3   DiastolicBloodPressure  10000 non-null  int64  
 4   TricepsThickness        10000 non-null  int64  
 5   SerumInsulin            10000 non-null  int64  
 6   BMI                     10000 non-null  float64
 7   DiabetesPedigree        10000 non-null  float64
 8   Age                     10000 non-null  int64  
 9   Diabetic                10000 non-null  int64  
dtypes: float64(2), int64(8)
memory usage: 781.4 KB


In [52]:
df.head()

,PatientID,Pregnancies,PlasmaGlucose,DiastolicBloodPressure,TricepsThickness,SerumInsulin,BMI,DiabetesPedigree,Age,Diabetic
0,1354778,0,171,80,34,23,43.509726,1.213191,21,0
1,1147438,8,92,93,47,36,21.240576,0.158365,23,0
2,1640031,7,115,47,52,35,41.511523,0.079019,23,0
3,1883350,9,103,78,25,304,29.582192,1.282870,43,1
4,1424119,1,85,59,27,35,42.604536,0.549542,22,0


In [32]:
import datetime

dt = datetime.datetime(2026, 10, 2, 18, 32, 48)
experiments = mlflow.search_experiments(filter_string=f"creation_time > {int(dt.timestamp())}")
for experiment in experiments:
    print(experiment.name)

prepare_image
azureml-examples
model
sdk_job_data_from_registry
heart-condition-classifier
heart-disease-classifier
Default
heart-classifier-sample
model-management-examples
mlflow-experiment
quickstart
heart-classifier-kr8n0-4595h
heart-classifier-sdloy
hello-batch-pzhot
Diabetes_Decision_Example_Model_Training_1790803444
bluff_experiment_studio_pipeline
hello-world-parallel-job
pipeline_samples
diabetes-training
sweep-diabetes
pipeline_diabetes
